# Modèle GPT2 sur SIEVE — variante output_0

Ce notebook entraîne deux modèles sur **une seule variante de données : `output_0`**.

Modèle : openai-community/gpt2"   

**Fichiers attendus :** `output_0_X_train.csv`, `output_0_X_val.csv` et `output_0_X_test.csv` dans `processed_data/` (ou `../processed_data/`), avec les colonnes `log` et `label`.



In [1]:
import os
import csv
import json
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("Appareil utilisé :", DEVICE)

VARIANTE = "output_0"

MODELES = [ 
    "openai-community/gpt2"         
]

BATCH_SIZE = 8
EPOCHS = 1
MAX_LENGTH = 128
LEARNING_RATE = 2e-5

os.makedirs("resultats_gpt", exist_ok=True)


Appareil utilisé : mps


In [2]:
DATA_DIR = next(
    (
        dossier
        for dossier in (Path("../processed_data"), Path("processed_data"))
        if (dossier / f"{VARIANTE}_X_train.csv").is_file()
    ),
    None,
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Impossible de trouver processed_data/output_0_X_train.csv. "
        "Vérifie l'emplacement des fichiers préparés."
    )


def charger_split(path):
    lignes = []

    with path.open("r", encoding="utf-8", newline="") as fichier:
        lecteur = csv.reader(fichier)
        entete = next(lecteur, None)

        if entete != ["log", "label"]:
            raise ValueError(f"En-tête inattendu dans {path} : {entete}")

        for numero, ligne in enumerate(lecteur, start=2):
            if not ligne:
                continue
            if len(ligne) < 2:
                raise ValueError(f"Ligne {numero} invalide dans {path}")

            # Les virgules éventuelles dans le log sont regroupées.
            log = ",".join(ligne[:-1]).strip()

            try:
                label = int(ligne[-1].strip())
            except ValueError as exc:
                raise ValueError(
                    f"Label invalide à la ligne {numero} dans {path}"
                ) from exc

            lignes.append({"log": log, "label": label})

    return pd.DataFrame(lignes, columns=["log", "label"])


train_df = charger_split(DATA_DIR / f"{VARIANTE}_X_train.csv")
val_df = charger_split(DATA_DIR / f"{VARIANTE}_X_val.csv")
test_df = charger_split(DATA_DIR / f"{VARIANTE}_X_test.csv")

# Vérifier que les identifiants de classes sont contigus de 0 à C-1.
classes = sorted(train_df["label"].unique().tolist())
if classes != list(range(len(classes))):
    raise ValueError(
        "Les labels doivent être encodés de 0 à nombre_de_classes - 1. "
        "Vérifie l'encodage réalisé dans le prétraitement."
    )

NUM_CLASSES = len(classes)
print("Train :", train_df.shape)
print("Validation :", val_df.shape)
print("Test :", test_df.shape)
print("Nombre de classes :", NUM_CLASSES)


Train : (24500, 2)
Validation : (5250, 2)
Test : (5250, 2)
Nombre de classes : 30


In [3]:
class LogsDataset(Dataset):
    def __init__(self, dataframe):
        self.textes = dataframe["log"].astype(str).tolist()
        self.labels = dataframe["label"].astype(int).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.textes[idx], self.labels[idx]


def creer_collate_fn(tokenizer):
    def collate_fn(batch):
        textes, labels = zip(*batch)

        encodages = tokenizer(
            list(textes),
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        encodages["labels"] = torch.tensor(labels, dtype=torch.long)

        return encodages

    return collate_fn


def evaluer_modele(model, loader):
    model.eval()
    vraies, predictions, pertes = [], [], []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            sorties = model(**batch)

            pertes.append(sorties.loss.item())
            pred = torch.argmax(sorties.logits, dim=1)

            vraies.extend(batch["labels"].cpu().numpy())
            predictions.extend(pred.cpu().numpy())

    accuracy = accuracy_score(vraies, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        vraies, predictions, average="macro", zero_division=0
    )

    return (
        float(np.mean(pertes)),
        accuracy,
        precision,
        recall,
        f1,
        vraies,
        predictions
    )


## Entraînement des deux modèles sur output_0

In [4]:
MODELES = [  "openai-community/gpt2" ]

resultats = []

train_dataset = LogsDataset(train_df)
val_dataset = LogsDataset(val_df)
test_dataset = LogsDataset(test_df)

for nom_modele in MODELES:
    nom_dossier = nom_modele.split("/")[-1]

    print("\n" + "=" * 70)
    print("MODÈLE :", nom_modele)
    print("=" * 70)

    tokenizer = AutoTokenizer.from_pretrained(nom_modele)

    # GPT-2 n’a pas de token de padding par défaut.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    model = AutoModelForSequenceClassification.from_pretrained(
        nom_modele,
        num_labels=NUM_CLASSES
    ).to(DEVICE)

    model.config.pad_token_id = tokenizer.pad_token_id

    collate_fn = creer_collate_fn(tokenizer)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        collate_fn=collate_fn
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_fn
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_fn
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE
    )

    meilleur_f1_val = -1
    meilleur_dossier = f"resultats_gpt/{nom_dossier}_meilleur"

    for epoch in range(EPOCHS):
        model.train()
        pertes_train = []

        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            optimizer.zero_grad()

            sorties = model(**batch)
            loss = sorties.loss

            loss.backward()
            optimizer.step()

            pertes_train.append(loss.item())

        val_loss, val_acc, val_precision, val_recall, val_f1, _, _ = (
            evaluer_modele(model, val_loader)
        )

        print(
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"Train loss: {np.mean(pertes_train):.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val accuracy: {val_acc:.4f} | "
            f"Val macro-F1: {val_f1:.4f}"
        )

        if val_f1 > meilleur_f1_val:
            meilleur_f1_val = val_f1

            model.save_pretrained(meilleur_dossier)
            tokenizer.save_pretrained(meilleur_dossier)

    # Recharger le meilleur modèle
    del model

    if DEVICE.type == "mps":
        torch.mps.empty_cache()

    model = AutoModelForSequenceClassification.from_pretrained(
        meilleur_dossier
    ).to(DEVICE)

    test_loss, test_acc, test_precision, test_recall, test_f1, y_true, y_pred = (
        evaluer_modele(model, test_loader)
    )

    print(
        f"TEST | Accuracy: {test_acc:.4f} | "
        f"Macro-F1: {test_f1:.4f}"
    )

    print(classification_report(y_true, y_pred, zero_division=0))

    # Sauvegarder le modèle final
    dossier_final = f"resultats_gpt/{nom_dossier}_final"

    model.save_pretrained(dossier_final)
    tokenizer.save_pretrained(dossier_final)

    resultats.append({
        "modele": nom_modele,
        "variante_dataset": VARIANTE,
        "accuracy": test_acc,
        "precision_macro": test_precision,
        "recall_macro": test_recall,
        "f1_macro": test_f1,
        "test_loss": test_loss
    })

    # Sauvegarder les résultats après chaque modèle
    pd.DataFrame(resultats).to_csv(
        "resultats_gpt/comparaison_gpt_output_0.csv",
        index=False
    )

    del model, tokenizer, train_loader, val_loader, test_loader

    if DEVICE.type == "mps":
        torch.mps.empty_cache()

print("\nEntraînement GPT-2 terminé.")


MODÈLE : openai-community/gpt2


W1117 21:31:44.392000 70582 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: openai-community/gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/1 | Train loss: 0.4959 | Val loss: 0.0598 | Val accuracy: 0.9701 | Val macro-F1: 0.9698


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

TEST | Accuracy: 0.9705 | Macro-F1: 0.9702
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       175
           1       0.99      1.00      1.00       175
           2       0.93      0.97      0.95       175
           3       0.95      0.94      0.95       175
           4       0.94      0.95      0.95       175
           5       1.00      1.00      1.00       175
           6       1.00      0.77      0.87       175
           7       1.00      1.00      1.00       175
           8       0.81      1.00      0.90       175
           9       1.00      1.00      1.00       175
          10       1.00      0.77      0.87       175
          11       0.81      1.00      0.90       175
          12       1.00      1.00      1.00       175
          13       1.00      1.00      1.00       175
          14       1.00      1.00      1.00       175
          15       1.00      1.00      1.00       175
          16       1.00      1.00     

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Entraînement GPT-2 terminé.


## Comparaison finale

In [ ]:
comparaison = pd.DataFrame(resultats)
display(comparaison)
comparaison.to_csv(
    "resultats_gpt/comparaison_gpt_output_0.csv",
    index=False
)
